# Tier 2 Full-Data — GPU Runtime T4 (FT-Transformers + FT-CUR)

Roda **5 modelos × 6 datasets × 30 seeds** com **N completo** (sem cap):
- **FT-Transformer baselines (4)**: Softmax, TopK, Entmax, Sparsemax
- **FT-CUR (Nyströmformer)**: atenção inter-instâncias comprimida

Datasets (N completo): ADULT (45k), CREDIT (30k), BANK (45k), TELCO (7k), SHOPPERS (12k), HIGGS50K (50k).

**Antes de rodar:** `Runtime → Change runtime type → T4 GPU` (ou A100 se Pro).

**Notebook complementar**: `tier2_colab_cpu.ipynb` (XGBoost + Nyström-SVM no CPU runtime — não consome tempo de GPU).

In [ ]:
# ── Célula 1: GPU check ─────────────────────────────────────────────────────
!nvidia-smi -L
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}')
    print(f'VRAM: {p.total_memory / 1e9:.1f} GB')
    print(f'CC: ({p.major}, {p.minor})')
else:
    print('⚠️ GPU não disponível — Runtime → Change runtime type → GPU')

In [ ]:
# ── Célula 2: Clonar do GitHub ──────────────────────────────────────────────
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'\nDiretório atual: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q numpy scipy scikit-learn pandas optuna xgboost xlrd pyarrow

import numpy, scipy, sklearn, torch, optuna, xgboost
print(f'numpy {numpy.__version__} | scipy {scipy.__version__} | '
      f'sklearn {sklearn.__version__} | torch {torch.__version__}')
print(f'optuna {optuna.__version__} | xgboost {xgboost.__version__}')

In [ ]:
# ── Célula 4: Baixar datasets ──────────────────────────────────────────────
!python scripts/download_data.py --tier 2
!ls -lh data/raw/ | head -15

In [ ]:
# ── Célula 5: Montar Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier2'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/tuning', exist_ok=True)
print(f'Drive em: {DRIVE_PATH}')
!ls -lh '{DRIVE_PATH}' 2>/dev/null

In [ ]:
# ── Célula 6: Restaurar progresso (resume) ──────────────────────────────────
import shutil
from pathlib import Path

drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)
(local_results / 'tuning').mkdir(exist_ok=True)

# Arquivo dedicado GPU (separado do CPU notebook)
for fname in ['tier2_full_gpu.json',
              'tuning/best_params_tier2_full_gpu.json']:
    src = drive_results / fname
    dst = local_results / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'✓ Restaurado: {fname}')
    else:
        print(f'• Começando do zero: {fname}')

In [ ]:
# ── Célula 7: Sync para Drive em background (a cada 5 min) ─────────────────
%%writefile /content/sync_to_drive.sh
#!/bin/bash
while true; do
    sleep 300
    cp -u /content/sparse-lssvm-transformers-study/results/tier2_full_gpu.json \
          "$1/tier2_full_gpu.json" 2>/dev/null
    cp -u /content/sparse-lssvm-transformers-study/results/tuning/best_params_tier2_full_gpu.json \
          "$1/tuning/best_params_tier2_full_gpu.json" 2>/dev/null
done

In [ ]:
import subprocess
sync_proc = subprocess.Popen(['bash', '/content/sync_to_drive.sh', DRIVE_PATH])
print(f'Sync rodando em background (PID {sync_proc.pid}) — salva a cada 5 min')

In [ ]:
# ── Célula 8: Rodar experimentos GPU (4 FT baselines + FT-CUR) ──────────────
# 5 modelos × 6 datasets × 30 seeds = 900 runs
# --no-cap: dataset COMPLETO (sem subsample)
# --models-group colab_gpu: 4 FT-Transformer baselines + FT-CUR
# Em T4 (16GB): ~10-12h. Em A100: ~4-5h.

!python scripts/run_tier2_n5000.py \
    --seeds 30 \
    --trials 20 \
    --folds 3 \
    --models-group colab_gpu \
    --no-cap \
    --output-file results/tier2_full_gpu.json \
    --params-file results/tuning/best_params_tier2_full_gpu.json

In [ ]:
# ── Célula 9: Salvar no Drive ───────────────────────────────────────────────
import shutil, signal
from pathlib import Path

try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

drive_dest = Path(DRIVE_PATH)
(drive_dest / 'tuning').mkdir(exist_ok=True)

for fname in ['tier2_full_gpu.json',
              'tuning/best_params_tier2_full_gpu.json']:
    src = Path('results') / fname
    dst = drive_dest / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'✓ Salvo: {dst.name} ({src.stat().st_size / 1024:.1f} KB)')

print(f'\nConteúdo do Drive:')
!ls -lh '{DRIVE_PATH}'

In [ ]:
# ── Célula 10: Análise rápida ───────────────────────────────────────────────
import json, numpy as np
from collections import defaultdict

results = json.load(open('results/tier2_full_gpu.json'))
print(f'Total runs: {len(results)}  ({sum(1 for r in results if r.get("status") == "ok")} OK)\n')

DATASETS = ['ADULT', 'CREDIT', 'BANK', 'TELCO', 'SHOPPERS', 'HIGGS50K']
scores = defaultdict(lambda: defaultdict(list))
for r in results:
    if r.get('status') != 'ok': continue
    m = r.get('model_variant') or r.get('model')
    scores[m][r['dataset']].append(r.get('f1_macro', float('nan')))

print(f'{"Modelo":<28}' + ''.join(f'{d:>10}' for d in DATASETS) + f'{"Média":>10}')
print('─' * 100)
for m in sorted(scores.keys()):
    vals = [np.mean(scores[m].get(d, [float("nan")])) for d in DATASETS]
    mean = np.nanmean(vals)
    print(f'{m:<28}' + ''.join(f'{v:>10.4f}' if not np.isnan(v) else f'{"-":>10}'
                                for v in vals) + f'{mean:>10.4f}')